# Agent training

In [ ]:
# ----- HELPER FUNCTIONS ----

# ----- Mini helpers ----

import subprocess

def get_git_version():
    try:
        return subprocess.check_output(
            ["git", "describe", "--tags", "--abbrev=0"],
            stderr=subprocess.DEVNULL
        ).decode().strip()
    except Exception:
        return "unknown"

def make_model_path(run_dir, model_id=None): 
    if model_id is None:
        model_id = run_dir
    return f"runs/{run_dir}/{model_id}.zip"

# ----- Model comparison ----

from training_utils import further_train_model, evaluate_model

def compare_all_models(algo_dict):
    """
    algo_dict = {
        "RANDOM": None,
        "DQN": "runs/v0.7.4_circle_DQN_2024-12-10_13-42-17.zip"
    }
    """
    for algo, model in algo_dict.items():
        evaluate_model(
            scenario="all_random",
            algorithm=algo,
            version_tag=get_git_version(),
            reward_strategy="basic",
            street_network="straight100km",
            n_vehicles=15,
            n_episodes= 50,
            render_mode=None,
            model_load_path=model,
            random_seed=123
        )


In [ ]:
# --- Benchmark evaluation ---
from training_utils import evaluate_model
from datetime import datetime, timezone

version_tag = get_git_version()
algorithm = "GREEDY"
n_vehicles = 2
n_nmevs = 0
n_episodes = 3
timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")


def benchmark_eval():
    evaluate_model(
    scenario="all_random",
    algorithm=algorithm, 
    version_tag=version_tag, 
    reward_strategy="basic", 
    street_network="straight100km", 
    n_vehicles=n_vehicles, 
    n_nmevs=n_nmevs,
    n_episodes=n_episodes, 
    render_mode=None, 
    model_load_path=None, 
    random_seed=123,
    use_wandb=True,
    wandb_entity="evcs-rl",
)
    
    
import cProfile
benchmark_file_name = "_".join(["output_eval", version_tag, f"{n_vehicles}MEV", f"{n_nmevs}NMEV", algorithm, f"{n_episodes}eps", timestamp])
benchmark_file = f'{benchmark_file_name}.pstats'
cProfile.run('benchmark_eval()', benchmark_file)

In [ ]:
# manually configure model load path
model_dir = "2026-02-17_18-04-15_v0.7.9_basic_straight100Test_PPO"
model_id = "2026-02-17_18-04-15_v0.7.9_basic_straight100Test_PPO"
model_path = make_model_path(model_dir, model_id)

In [ ]:
from training_utils import evaluate_model

evaluate_model(
    scenario="all_random",
    algorithm="PPO", 
    version_tag=get_git_version(), 
    reward_strategy="basic", 
    street_network="straight100km", 
    n_vehicles=2, 
    n_nmevs=0,
    n_episodes=5, 
    render_mode=None, 
    model_load_path=model_path, 
    random_seed=123,
    use_wandb=True,
    wandb_entity="evcs-rl",
)

In [ ]:
from training_utils import train_and_evaluate

# filepath_list = [
#     "runs/2025-04-04_12-24-57_v0.7.5_basic_circleSameSOC_PPO/evaluation/metrics2025-04-04_13-03-37.json",
#     "runs/2025-04-04_13-04-54_v0.7.5_basic_circleSameSOC_RANDOM/evaluation/metrics2025-04-04_13-04-53.json",
#     "runs/2025-04-04_13-06-12_v0.7.5_basic_circleSameSOC_GREEDY/evaluation/metrics2025-04-04_13-06-11.json"
# ]

model_path, filepath_list = train_and_evaluate(
    scenario="BASt",
    algorithm="PPO", 
    policy="MultiInputPolicy", 
    version_tag=get_git_version(), 
    reward_strategy="basic", 
    street_network="straight100Test_500MEV_0NMEV", 
    n_vehicles=500, 
    n_nmevs=0, 
    n_steps=10000, 
    execution_context="local"
)

In [ ]:
algo_dict = {
    # "RANDOM": None,
    # "GREEDY": None,
    "A2C": make_model_path("2025-03-01_14-19-38_v0.7.4_basic_circle_A2C"),
    "A2C": make_model_path("2025-03-01_14-50-17_v0.7.4_shaping_circle_A2C"),
    "PPO": make_model_path("2025-03-01_13-33-24_v0.7.4_basic_circle_PPO"),
    "PPO": make_model_path("2025-03-01_12-42-26_v0.7.4_shaping_circle_PPO"),
    "DQN": make_model_path("2025-03-01_15-35-16_v0.7.4_basic_circle_DQN"),
    "DQN": make_model_path("2025-03-01_16-00-51_v0.7.4_shaping_circle_DQN"),
}

compare_all_models(algo_dict)

In [ ]:
# Train new model
from training_utils import train_model

model_path = train_model(
    scenario="all_random",
    algorithm="PPO", 
    policy="MultiInputPolicy", 
    version_tag=get_git_version(), 
    reward_strategy="basic", 
    street_network="straight100Test", 
    n_vehicles=2, n_steps=1, execution_context="local", random_seed=None, use_wandb=True, wandb_entity="evcs-rl")

In [ ]:
# Train new model
model_path = train_model(
    scenario="all_random",
    algorithm="DQN", 
    policy="MultiInputPolicy", 
    version_tag=get_git_version(), 
    reward_strategy="shaping", 
    street_network="circle", 
    n_vehicles=5, n_steps=50000, execution_context="local", random_seed=None)

In [ ]:
# manually configure model load path
run_dir = "v0.7.4_circle_DQN_2024-12-10_13-42-17"
model_id = "v0.7.4_circle_DQN_2024-12-10_13-42-17"
model_path = make_model_path(run_dir, model_id)

In [ ]:
# Train saved model further
model_path = further_train_model("PPO", get_git_version(), "basic", "circleSameSOC", n_vehicles=5, n_steps=100000, model_load_path=model_path)

In [ ]:
# Quick evaluation
# model_path = "runs/v0.3_circle_a2c_2024-09-18_23-58-11/v0.3_circle_a2c_2024-09-18_23-58-11.zip"
evaluate_model("DQN", get_git_version(), "noTime", "circle", n_vehicles=5, n_episodes=5, model_load_path=model_path)

In [ ]:
from stable_baselines3 import PPO, A2C, DQN
from environment import CustomEnv

# Observe execution of trained agent in GUI
env = CustomEnv(scenario_generator="all_random", render_mode="human", vehicles_to_spawn=5)
model = A2C.load(model_path, env)

num_steps = 5
observation, info = env.reset()
for t in range(num_steps):
        actions, _ = model.predict(observation, state=None, deterministic=False)
        observation, reward, terminated, truncated, info = env.step(actions)

env.close()

In [ ]:
# Random actions to compare with the agent
env = CustomEnv(scenario_generator="all_random", render_mode="human", vehicles_to_spawn=5)
observation, info = env.reset()
try:
    for _ in range(5):
        action = env.action_space.sample() # select a random action
        observation, reward, terminated, truncated, info = env.step(action)
        # if terminated or truncated:
            # observation, info = env.reset()
finally:        
    env.close()

### Benchmarks

In [ ]:
# --- Benchmark environment ---

from environment import CustomEnv

def benchmark_env(random_seed):
    env = CustomEnv(scenario_generator="all_random", render_mode=None, vehicles_to_spawn=15)
    # set seed for reproducability
    import random
    random.seed(random_seed) # needed for batteries of simulation class TODO: make this seedable via the env.seed of gymnasium
    observation, info = env.reset(seed=random_seed)
    env.action_space.seed(random_seed)
    try:
        for _ in range(200):
            action = env.action_space.sample() # select a random action
            observation, reward, terminated, truncated, info = env.step(action)
            if terminated or truncated:
                observation, info = env.reset()
    finally:
        env.close()
    
import cProfile
# cProfile.run('train_model("A2C", "MultiInputPolicy", "v0.3", "circle", n_vehicles=5, n_steps=10000)', 'output.pstats')
benchmark_file = 'output_v0.8.3.pstats'
cProfile.run('benchmark_env(random_seed=1)', benchmark_file)

In [ ]:
import pstats
from pstats import SortKey
p = pstats.Stats(benchmark_file)
p.sort_stats(SortKey.CUMULATIVE).print_stats()

# For a visual analysis use snakeviz in the terminal:
# snakeviz output_v0.8.3.pstats